# Laboratoria 11 - Kompleksowa Analiza Porównawcza: CNN vs ViT vs ViT z Early Convolutions

In [1]:
import torch
import math
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

## Przygotowanie danych i Zaawansowana Augmentacja
Klasyczne modele Vision Transformer (ViT) cierpią na tzw. głód danych. Wynika to z faktu, że w przeciwieństwie do sieci konwolucyjnych (CNN), ViT nie posiada wbudowanych założeń strukturalnych dotyczących obrazu (brakuje mu przestrzennego inductive bias). Model na początku treningu nie wie, że sąsiadujące ze sobą piksele tworzą spójne obiekty.

Z tego powodu trenowanie czystego ViT na małych zbiorach danych (takich jak CIFAR-10) bez agresywnej augmentacji prowadzi do drastycznego przeuczenia (overfittingu). Aby temu zapobiec, wprowadzamy nowoczesne techniki regularyzacji obrazu:
- Nowe API torchvision.transforms.v2 - zastępuje starszą wersję biblioteki. Działa szybciej i pozwala na tzw. augmentację wsadową (batch augmentation), co jest kluczowe dla zaawansowanych technik mieszania obrazów.
- MixUp - losowo dobiera dwa obrazy z paczki (batcha) i nakłada je na siebie (tworząc efekt „ducha” / półprzezroczystości). Etykiety również są mieszane proporcjonalnie. Zmusza to model do mniejszej pewności siebie i uczy go płynnych przejść między klasami (np. 70% kot, 30% pies).
- CutMix - wykracza poza proste nakładanie – wycina prostokątny fragment z jednego obrazu i wkleja go w drugi. Pomaga to modelowi skupiać się na wielu detalach naraz, a nie tylko na najbardziej widocznej cesze obiektu (np. uczy model rozpoznawać psa po łapach, gdy głowa zostanie zasłonięta fragmentem innego zdjęcia).

In [2]:
transform_train = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.RandomAffine(degrees=15, translate=(0.1, 0.1)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

transform_test = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

cutmix = v2.CutMix(num_classes=10)
mixup = v2.MixUp(num_classes=10)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

train_ds = CIFAR10(root='data', train=True, download=True, transform=transform_train)
test_ds = CIFAR10(root='data', train=False, download=True, transform=transform_test)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True) # Zwiększony batch_size
test_dl = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

Files already downloaded and verified
Files already downloaded and verified


## 1. Klasyczna Sieć Konwolucyjna (CNN)
Sieci konwolucyjne dominowały w wizji komputerowej ze względu na dwa fundamentalne założenia matematyczne wszyte w ich architekturę:
1. Lokalność (Locality): Filtry konwolucyjne operują na małych podgrupach pikseli (np. $3 \times 3$), zakładając, że najbliższe punkty niosą najważniejsze wspólne informacje.
2. Inwariantność na przesunięcia (Translation Equivariance): Ten sam filtr jest przesuwany po całym obrazie, co oznacza, że jeśli sieć nauczy się wykrywać krawędź w lewym górnym rogu, rozpozna ją również w prawym dolnym.

Pole recepcyjne (receptive field) w CNN rośnie stopniowo, warstwa po warstwie. Głębokie warstwy widzą cały obraz, ale pierwsze widzą jedynie mikro-detale.

In [3]:
class CNN(nn.Module):
  def __init__(self, out_features, kernel_size, channels_list, ifStride, input_size=(32, 32)):
    super().__init__()
    self.model = torch.nn.Sequential()
    self.model.add_module("in_conv", torch.nn.Conv2d(in_channels=3, out_channels=channels_list[0], kernel_size=1, padding=0))
    self.model.add_module("relu_1", torch.nn.ReLU())

    in_chan = channels_list[0]
    for i, out_chan in enumerate(channels_list):
      pad = kernel_size // 2
      if ifStride:
        self.model.add_module(f"conv_{i+1}", nn.Conv2d(in_channels=in_chan, out_channels=out_chan, kernel_size=kernel_size, stride=2, padding=pad))
        self.model.add_module(f"relu_{i+1}", nn.ReLU())
      else:
        self.model.add_module(f"conv_{i+1}", nn.Conv2d(in_channels=in_chan, out_channels=out_chan, kernel_size=kernel_size, stride=1, padding=pad))
        self.model.add_module(f"relu_{i+1}", nn.ReLU())
        self.model.add_module(f"max_pool_{i+1}", nn.MaxPool2d(kernel_size=2, stride=2))
      in_chan = out_chan

    num_layers = len(channels_list)
    img_h, img_w = input_size
    final_h = img_h // (2 ** num_layers)
    final_w = img_w // (2 ** num_layers)

    linear_in_features = channels_list[-1] * final_h * final_w
    self.model.add_module("flatten", torch.nn.Flatten())
    self.model.add_module("linear", nn.Linear(in_features=linear_in_features, out_features=out_features))

  def forward(self,x):
    return self.model(x)

## 2. Architektura Vision Transformer (ViT)
Vision Transformer całkowicie odrzuca paradygmat konwolucyjny. Zamiast operować na lokalnych polach recepcyjnych, wprowadza Globalny Kontekst już od pierwszej warstwy za pomocą mechanizmu samouwagi.

### Wielogłowicowy Mechanizm Samouwagi (Multi-Head Self-Attention - MSA)
Każdy element wejściowy (token) generuje trzy wektory: Zapytanie ($Q$), Klucz ($K$) oraz Wartość ($V$). Matematyczna reprezentacja pojedynczej głowy uwagi opisana jest wzorem:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Gdzie $d_k = \frac{d_{model}}{\text{num\_heads}}$ oznacza wymiar pojedynczej głowy. Podział przez $\sqrt{d_k}$ jest kluczowy – zapobiega on generowaniu ekstremalnie dużych wartości iloczynu skalarnego, co prowadziłoby do spłaszczenia gradientów w funkcji softmax (problem saturacji). Koszt obliczeniowy mechanizmu uwagi rośnie kwadratowo $O(N^2)$ względem liczby łatek $N$.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def scaled_dot_product_attention(self, Q, K, V):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_probs = F.softmax(attn_scores, dim=-1)
        self.last_attn_probs = attn_probs
        output = torch.matmul(attn_probs, V)
        return output

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> torch.Tensor:
        batch_size = query.size(0)

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_output = self.scaled_dot_product_attention(Q, K, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(attn_output)

        return output

### Blok Feed-Forward i Przewaga Aktywacji GELU
Po warstwie MSA sygnał trafia do sieci pozycyjnej FFN. W architekturach typu Transformer preferuje się funkcję GELU (Gaussian Error Linear Unit) nad tradycyjnym ReLU. ReLU gwałtownie zeruje wszystkie wartości ujemne, co w bardzo głębokich sieciach prowadzi do problemu tzw. martwych neuronów (gradient trwale zanika). GELU modyfikuje sygnał w sposób płynny i probabilistyczny, pozwalając na przepływ małych wartości ujemnych, co stabilizuje proces optymalizacji.

In [5]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 2*d_model)
        self.fc2 = nn.Linear(2*d_model, d_model)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

### Kodowanie Pozycyjne (Positional Encoding)
Ponieważ mechanizm uwagi przetwarza wszystkie tokeny (łatki) jednocześnie (w sposób w pełni równoległy), sieć traci informację o geometrii obrazu. Permutacja (pomieszanie) łatek dałaby matematycznie ten sam wynik. Aby temu zapobiec, do wektorów cech dodaje się niezmiennicze wektory pozycji (wyznaczone za pomocą funkcji sinus i cosinus o różnych częstotliwościach), co pozwala sieci odzyskać orientację przestrzenną.

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        P_temp = torch.zeros((1, max_len, d_model))
        X = (torch.arange(max_len, dtype=torch.float32).reshape(-1, 1)
             / torch.pow(10000, torch.arange(0, d_model, 2, dtype=torch.float32) / d_model))
        P_temp[:, :, 0::2] = torch.sin(X)
        P_temp[:, :, 1::2] = torch.cos(X)

        self.register_buffer('P', P_temp)

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :]
        return self.dropout(X)

### Regularyzacja: Stochastic Depth (DropPath)
Głębokie modele Vision Transformer mają ogromną liczbę parametrów, co czyni je bardzo podatnymi na przeuczenie. Klasyczny Dropout (wyłączanie pojedynczych neuronów) często nie wystarcza w rozbudowanych blokach Transformatora. Dlatego stosujemy potężniejszą technikę – Stochastic Depth (czyli DropPath).
- Zamiast pojedynczych neuronów, algorytm losowo wyłącza całe ścieżki obliczeniowe (np. całą warstwę uwagi lub warstwę liniową) dla wybranych obrazów w danej paczce (batchu).
- Zmusza to model do budowania alternatywnych dróg przepływu informacji. Podczas treningu sieć zachowuje się jak zespół (ensemble) wielu płytszych modeli, co drastycznie poprawia generalizację na zbiorze testowym.
- Ścieżki, które przetrwały, są sztucznie "wzmacniane" (skalowane proporcjonalnie do prawdopodobieństwa odrzucenia), aby siła sygnału pozostała niezmienna podczas fazy testowej, gdy DropPath jest wyłączony.

In [7]:
def drop_path(x, drop_prob: float = 0., training: bool = False):
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    output = x.div(keep_prob) * random_tensor
    return output

class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)

### Bloki Kodera z Architekturą Pre-LayerNorm i DropPath
Współczesne modele ViT implementują architekturę Pre-LayerNorm (Pre-LN). Normalizacja warstwowa (LayerNorm) jest aplikowana przed blokami funkcjonalnymi (MSA i FFN), a wynik jest dodawany do pierwotnego wejścia za pomocą połączenia resztkowego (skip connection).

To właśnie w tych połączeniach resztkowych aplikujemy omówiony wcześniej mechanizm DropPath. Zaktualizowane równania przepływu sygnału wyglądają następująco:
$$x_{msa} = x + \text{DropPath}(\text{MSA}(\text{LayerNorm}(x)))$$
$$Wyjscie = x_{msa} + \text{DropPath}(\text{FFN}(\text{LayerNorm}(x_{msa})))$$

Zapewnia to bezpośrednią ścieżkę powrotną dla gradientów od ostatnich do pierwszych warstw sieci. Jeśli DropPath zdecyduje się wylosować odrzucenie danej ścieżki w kroku treningowym, cały skomplikowany blok uwagi lub FFN jest ignorowany, a sygnał przechodzi przez sieć w niezmienionej formie (jako czyste $x$).

In [8]:
class EncoderLayers(nn.Module):
    def __init__(self, d_model, num_heads, dropout, drop_path_rate=0.0):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()

    def forward(self, x):
        nx = self.norm1(x)
        attn_output = self.self_attn(nx, nx, nx)
        x = x + self.drop_path(self.dropout(attn_output))

        nx2 = self.norm2(x)
        ff_output = self.feed_forward(nx2)
        x = x + self.drop_path(self.dropout(ff_output))
        return x

###  Reprezentacja Wejściowa (ViT Embedding & Token [CLS])
Aby obraz dwuwymiarowy $H \times W \times C$ przesłać do jednowymiarowego Transformatora, poddaje się go procesowi linearyzacji:
1. Patchify Stem: Obraz jest dzielony na siatkę nienakładających się łatek o rozmiarze $P \times P$. W praktyce realizuje się to warstwą nn.Conv2d gdzie kernel_size = patch_size oraz stride = patch_size.
2. Liczba Tokenów ($N$): Wynosi dokładnie $N = \frac{H \cdot W}{P^2}$.
3. Token Klasyfikacji [CLS]: Dokładnie tak jak w modelach tekstowych BERT, na indeksie zero dokleja się sztuczny, wyuczalny wektor. Ponieważ mechanizm uwagi miesza informacje ze wszystkich tokenów, to właśnie końcowy stan wektora [CLS] posłuży do podjęcia ostatecznej decyzji klasyfikacyjnej. Łączna długość sekwencji wynosi $N + 1$.

In [9]:
class ViTEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768, dropout=0.1):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_encoder = PositionalEncoding(d_model=embed_dim, dropout=dropout, max_len=self.num_patches + 1)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                         # (B, D, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)                # (B, N, D)

        cls_tokens = self.cls_token.expand(B, -1, -1)   # (B, 1, D)
        x = torch.cat((cls_tokens, x), dim=1)           # (B, N+1, D)

        x = self.pos_encoder(x)
        return x                                        # (B, N+1, D)

### Klasyfikator i Agregacja Globalna (Architektura ViTTransformer)
Główna klasa ViTTransformer łączy wszystkie omówione wcześniej komponenty w jedną całość. Warto zwrócić uwagę na dwa kluczowe mechanizmy decyzyjne zastosowane w tej architekturze:

- **Agregacja przez token [CLS]** - zamiast uśredniać wyniki ze wszystkich tokenów obrazu (jak w przypadku Global Average Pooling w sieciach CNN), ViT polega wyłącznie na specjalnym wektorze klasyfikacyjnym [CLS]. Dzięki mechanizmowi samouwagi w każdym bloku kodera, token [CLS] systematycznie zbiera i agreguje najważniejsze informacje ze wszystkich pozostałych łatek obrazu. Ostateczna decyzja klasyfikatora bazuje tylko na tym jednym, zaktualizowanym wektorze.

- **Liniowe skalowanie DropPath** - prawdopodobieństwo wyłączenia ścieżki (drop_path_rate) nie jest stałe dla każdego bloku. W kodzie zastosowano regułę rosnącą liniowo (od zera do wartości docelowej). Wczesne warstwy uczą się uniwersalnych cech (ich odrzucenie zdestabilizowałoby trening), podczas gdy głębokie warstwy są bardziej skłonne do przeuczenia (overfittingu) – dlatego to właśnie je odrzucamy z większym prawdopodobieństwem.

In [10]:
class ViTTransformer(nn.Module):
    def __init__(self, num_classes=10, num_layers=12, img_size=224, patch_size=16, in_channels=3, embed_dim=768, num_heads=12, dropout=0.1, drop_path_rate=0.1):
        super().__init__()

        self.embedding = ViTEmbedding(img_size=img_size, patch_size=patch_size, in_channels=in_channels, embed_dim=embed_dim, dropout=dropout)

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, num_layers)]
        self.encoder_layers = nn.ModuleList([
            EncoderLayers(d_model=embed_dim, num_heads=num_heads, dropout=dropout, drop_path_rate=dpr[i]) for i in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        for encoder in self.encoder_layers:
            x = encoder(x)

        cls_token = x[:, 0]
        cls_token = self.final_norm(cls_token)
        return self.classifier(cls_token)

## 3. Early Convolutions w ViT (Modyfikacja ViT_C)
Oryginalna metoda przygotowania danych za pomocą twardego cięcia obrazu (patchify stem o dużym filtrze np. $16 \times 16$) rodzi poważne problemy optymalizacyjne:
- Modele są ekstremalnie wrażliwe na zmiany hiperparametrów.
- Zmiana optymalizatora z zaawansowanego AdamW na klasyczny SGD wywołuje całkowitą zapaść wyników (lub błędy NaN wynikające z rozbiegania się gradientów).

Zastąpienie pojedynczej, potężnej konwolucji minimalistycznym blokiem złożonym z kilku (5-7) tradycyjnych warstw konwolucyjnych $3 \times 3$, które nakładają się na siebie, są stabilizowane przez BatchNorm2d i aktywowane przez ReLU. Wprowadza to ładodniejsze obciążenie indukcyjne (wczesne wychwytywanie lokalnych krawędzi i tekstur) na samym początku sieci. Złożoność obliczeniowa takiego bloku (EarlyConvStem) odpowiada kosztowi dokładnie jednego bloku transformatora. Dlatego, aby zachować równe warunki porównawcze (ta sama liczba operacji zmiennoprzecinkowych FLOPs), w modelach ViT_C usuwa się jedną warstwę encodera (zamiast 12 zostaje 11).

### Podstawowy Blok Konwolucyjny (Conv2DNormAct)
Zanim zbudujemy cały moduł wejściowy (Stem), potrzebujemy powtarzalnego klocka. W przeciwieństwie do czystego ViT, który wykorzystuje tylko jedną, agresywną warstwę liniową, tutaj wprowadzamy klasyczne podejście z sieci CNN.
- bias=False - w warstwie konwolucyjnej wyłączamy obciążenie (bias), ponieważ zaraz po niej następuje BatchNorm2d. Normalizacja wsadowa i tak wyzerowałaby ten bias (odejmując średnią), więc jego obecność to tylko niepotrzebne marnowanie pamięci i mocy obliczeniowej.
- inplace=True we wzbudzeniu - funkcja ReLU podmienia wartości bezpośrednio w pamięci tensora, co oszczędza VRAM podczas treningu.

In [11]:
class Conv2DNormAct(nn.Module):
    def __init__(self, in_chan: int, out_chan: int, kernel: int, stride: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_chan, out_chan, kernel, stride, padding=1, bias=False),
            nn.BatchNorm2d(out_chan),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

### Wczesny Moduł Konwolucyjny (EarlyConvStem)
Ten moduł całkowicie zastępuje tzw. Patchify Stem (cięcie obrazu na łatki jednym filtrem).Dla obrazów wejściowych o rozmiarze $32 \times 32$ (np. CIFAR-10), ten moduł stopniowo redukuje rozdzielczość przestrzenną (za pomocą stride=2), jednocześnie zwiększając liczbę kanałów (głębokość mapy cech). Matematycznie działa to tak:
1. Rozpoczyna od obrazu $32 \times 32$.
2. Pierwszy stride=2 redukuje siatkę do $16 \times 16$.
3. Drugi stride=2 redukuje ją do $8 \times 8$.

Otrzymana siatka $8 \times 8$ daje dokładnie $64$ tokeny przestrzenne. Jest to dokładny odpowiednik zastosowania w klasycznym ViT łatek o rozmiarze $4 \times 4$ ($32 / 4 = 8$).

In [12]:
class EarlyConvStem(nn.Module):
    def __init__(self, in_channels=3, embed_dim=768):
        super().__init__()

        self.stem = nn.Sequential(
            Conv2DNormAct(in_channels, 64, kernel=3, stride=1),   # 32x32 -> 32x32
            Conv2DNormAct(64, 128, kernel=3, stride=2),          # 32x32 -> 16x16
            Conv2DNormAct(128, 128, kernel=3, stride=1),         # 16x16 -> 16x16
            Conv2DNormAct(128, 256, kernel=3, stride=2),         # 16x16 -> 8x8
            Conv2DNormAct(256, 256, kernel=3, stride=1),         # 8x8 -> 8x8
            Conv2DNormAct(256, 512, kernel=3, stride=1),         # 8x8 -> 8x8
            nn.Conv2d(512, embed_dim, kernel_size=1, stride=1)   # 8x8 -> 8x8
        )

    def forward(self, x):
        return self.stem(x)

### Główny Model Hybrydowy (ViTEarlyConvTransformer)
Oto punkt kulminacyjny całego projektu, w którym łączymy potęgę lokalnego przetwarzania obrazu (CNN) z globalnym rozpoznawaniem kontekstu (ViT). W tej klasie kluczowe są dwa aspekty – transformacja wymiarów oraz nowo wprowadzona zaawansowana regularyzacja:

- **Przejście z CNN do ViT** - najważniejszą operacją wewnątrz funkcji forward jest przejście z czterowymiarowych map cech wygenerowanych przez sploty splotowe (Batch, Kanały, Wysokość, Szerokość) na sekwencję punktów danych, którą rozumie Transformer. Robimy to za pomocą operacji x.flatten(2).transpose(1, 2), która spłaszcza wymiary przestrzenne (H i W) w jeden wektor łatek, a następnie zamienia miejscami wymiar sekwencji z wymiarem osadzenia (embed_dim).

- **Dostosowanie DropPath (drop_path_rate)** - podobnie jak w czystym Transformerze, model hybrydowy został wzbogacony o mechanizm Stochastic Depth. Wprowadzenie parametru drop_path_rate i wyznaczenie listy dpr pozwala na stopniowe zwiększanie rygoru regularyzacji. Ponieważ początkowe warstwy splotowe (EarlyConvStem) wyciągają niskopoziomowe cechy (krawędzie, kształty), głębokie warstwy transformatora bez rosnącego DropPath błyskawicznie zaczęłyby dopasowywać się do szumu w małym zbiorze CIFAR-10.

In [13]:
class ViTEarlyConvTransformer(nn.Module):
    def __init__(self, num_classes=10, num_layers=12, img_size=224, patch_size=16, in_channels=3, embed_dim=768, num_heads=12, dropout=0.1, drop_path_rate=0.1):
        super().__init__()

        self.num_patches = (img_size // patch_size) ** 2
        self.embedding_stem = EarlyConvStem(in_channels=in_channels, embed_dim=embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_encoder = PositionalEncoding(d_model=embed_dim, dropout=dropout, max_len=self.num_patches + 1)

        # Liniowy wzrost prawdopodobieństwa DropPath (Stochastic Depth Decay Rule)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, num_layers)]

        self.encoder_layers = nn.ModuleList([
            EncoderLayers(d_model=embed_dim, num_heads=num_heads, dropout=dropout, drop_path_rate=dpr[i]) for i in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.embedding_stem(x)                     # (B, embed_dim, H/16, W/16)

        x = x.flatten(2).transpose(1, 2)               # (B, num_patches, embed_dim)

        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat((cls_tokens, x), dim=1)          # (B, num_patches+1, embed_dim)

        x = self.pos_encoder(x)

        for encoder in self.encoder_layers:
            x = encoder(x)

        cls_token = x[:, 0]
        cls_token = self.final_norm(cls_token)
        return self.classifier(cls_token)

## Pętla Trenująca i Optymalizacja Hiperparametrów
Proces optymalizacji głębokich Transformerów wymaga zastosowania zaawansowanych technik obliczeniowych i numerycznych, które stabilizują proces uczenia:

- **Mieszana Precyzja (torch.amp.autocast)** – część obliczeń wykonywana jest w formacie FP16 (zamiast FP32), co drastycznie redukuje zużycie pamięci VRAM oraz przyspiesza trening na nowoczesnych GPU.

- **Skalowanie Gradientów (GradScaler)** – zapobiega zjawisku niedomiaru (ang. underflow) dla małych wartości gradientów w formacie FP16.

- **Obcinanie Normy Gradientu (clip_grad_norm_)** – jeśli norma gradientu przekroczy zadaną wartość (np. max_norm=5.0), zostaje ona przeskalowana w dół. Chroni to model przed tzw. eksplozją gradientu, która jest powszechna podczas treningu niestabilnych sieci typu ViT.

- **Obsługa miękkich etykiet (Soft Labels)** – techniki takie jak MixUp i CutMix nie przypisują obrazom jednej klasy (np. 0 lub 1), lecz wektor prawdopodobieństwa (np. [0.7, 0.3]). Funkcja count_correct została rozbudowana o obsługę wielowymiarowych etykiet przy użyciu torch.argmax, co pozwala poprawnie monitorować dokładność (accuracy) pomimo zmiksowanego wektora docelowego.

- **Augmentacja wewnątrz pętli (cutmix_or_mixup)** – mieszanie danych odbywa się dynamicznie w locie na poziomie każdego mini-batcha przed wysłaniem danych na GPU.

- **Harmonogramowanie Per-Step (scheduler.step())** – krok optymalizatora i zmiana współczynnika uczenia (learning rate) są wywoływane po każdym przejściu batcha, a nie raz na epokę. Jest to kluczowe dla zaawansowanych strategii takich jak OneCycleLR, które dynamicznie manipulują tempem uczenia wewnątrz pojedynczej epoki, zapobiegając utknięciu w lokalnych minimach.

In [14]:
def count_correct(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    preds = torch.argmax(y_pred, dim=1)
    if y_true.ndim > 1:
        y_true = torch.argmax(y_true, dim=1)
    return (preds == y_true).float().sum()

def validate(model: nn.Module, loss_fn: torch.nn.CrossEntropyLoss, dataloader: DataLoader) -> tuple[float, float]:
    loss = 0
    correct = 0
    all_samples = 0
    for X_batch, y_batch in dataloader:
        X_batch_cuda = X_batch.cuda()
        y_batch_cuda = y_batch.cuda()

        y_pred = model(X_batch_cuda)
        batch_size = len(y_pred)
        all_samples += batch_size

        loss += loss_fn(y_pred, y_batch_cuda).item() * batch_size
        correct += count_correct(y_pred, y_batch_cuda)
    return loss / all_samples, float(correct) / all_samples

def fit(
    model: nn.Module, optimiser: optim.Optimizer, scheduler,
    loss_fn: torch.nn.CrossEntropyLoss, train_dl: DataLoader,
    val_dl: DataLoader, epochs: int, print_metrics: bool = True
):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{epochs}"):
            X_batch, y_batch = cutmix_or_mixup(X_batch, y_batch)

            X_batch_cuda = X_batch.cuda()
            y_batch_cuda = y_batch.cuda()

            with torch.amp.autocast(device_type='cuda'):
                y_pred = model(X_batch_cuda)
                loss = loss_fn(y_pred, y_batch_cuda)

            scaler.scale(loss).backward()
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimiser)
            scaler.update()
            optimiser.zero_grad()

            if scheduler is not None:
                scheduler.step()

        if print_metrics:
            model.eval()
            with torch.no_grad():
                train_loss, train_acc = validate(model=model, loss_fn=loss_fn, dataloader=train_dl)
                val_loss, val_acc = validate(model=model, loss_fn=loss_fn, dataloader=val_dl)

                history['train_loss'].append(train_loss)
                history['train_acc'].append(train_acc)
                history['val_loss'].append(val_loss)
                history['val_acc'].append(val_acc)

                print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    return history

def plot_metrixes(all_histories):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    ax_train_loss = axes[0, 0]
    ax_val_loss = axes[0, 1]
    ax_train_acc = axes[1, 0]
    ax_val_acc = axes[1, 1]

    for name, hist in all_histories.items():
        train_loss = [float(x) for x in hist['train_loss']]
        val_loss = [float(x) for x in hist['val_loss']]
        train_acc = [float(x) for x in hist['train_acc']]
        val_acc = [float(x) for x in hist['val_acc']]

        ax_train_loss.plot(train_loss, label=name, linewidth=2)
        ax_val_loss.plot(val_loss, label=name, linewidth=2, linestyle='--')

        ax_train_acc.plot(train_acc, label=name, linewidth=2)
        ax_val_acc.plot(val_acc, label=name, linewidth=2, linestyle='--')

    ax_train_loss.set_title("Train Loss")
    ax_train_loss.set_xlabel("Epoch")
    ax_train_loss.set_ylabel("Loss")
    ax_train_loss.legend()
    ax_train_loss.grid(True, linestyle=':', alpha=0.6)

    ax_val_loss.set_title("Validation Loss")
    ax_val_loss.set_xlabel("Epoch")
    ax_val_loss.set_ylabel("Loss")
    ax_val_loss.legend()
    ax_val_loss.grid(True, linestyle=':', alpha=0.6)

    ax_train_acc.set_title("Train Accuracy")
    ax_train_acc.set_xlabel("Epoch")
    ax_train_acc.set_ylabel("Accuracy")
    ax_train_acc.legend()
    ax_train_acc.grid(True, linestyle=':', alpha=0.6)

    ax_val_acc.set_title("Validation Accuracy")
    ax_val_acc.set_xlabel("Epoch")
    ax_val_acc.set_ylabel("Accuracy")
    ax_val_acc.legend()
    ax_val_acc.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

## Uruchomienie Eksperymentu Porównawczego
Podczas inicjalizacji modeli zwracamy szczególną uwagę na dobór optymalizatorów oraz strategii uczenia. Wprowadzamy tutaj kilka kluczowych technik, które są absolutnie niezbędne do ustabilizowania treningu wrażliwych architektur typu ViT:

- **Adam vs AdamW** - Dla tradycyjnej sieci CNN stosujemy standardowy algorytm Adam. Jednak dla czystego ViT oraz modelu hybrydowego (ViT_Early_Conv) bezwzględnie wymagany jest AdamW (wariant z poprawnym matematycznie odliczaniem zaniku wag – Weight Decay). Znacząco ogranicza to przeuczenie głębokich sieci. (Uwaga praktyczna: w bardzo zaawansowanych implementacjach wyłącza się parametr weight_decay dla warstw normalizacji oraz biasów, aby dodatkowo poprawić wydajność).

- **Wygładzanie Etykiet (Label Smoothing)** - Zastosowanie label_smoothing=0.1 w funkcji straty sprawia, że model nie dąży do 100% pewności swojej predykcji. Zamiast wektora [1, 0, 0] sieć uczy się rozkładu np. [0.9, 0.05, 0.05]. Zapobiega to nadmiernej pewności siebie (overconfidence) modelu i świetnie współpracuje z miękkimi etykietami pochodzącymi z augmentacji MixUp/CutMix.

- **Harmonogram OneCycleLR** - Modele Vision Transformer bardzo często "wybuchają" (tracą stabilność gradientów), jeśli zaczną trening z maksymalnym tempem uczenia. OneCycleLR chroni przed tym zjawiskiem: najpierw powoli "rozgrzewa" sieć (faza Warmup), a następnie płynnie wygasza krok uczenia zgodnie z krzywą kosinusoidalną (Cosine Annealing).

In [ ]:
EPOCHS = 50

configs = {
    "CNN": (
        CNN(10, 3, [16, 32, 64], False).cuda()
    ),
    "VIT_base": (
        ViTTransformer(img_size=32, patch_size=4, drop_path_rate=0.1).cuda()
    ),
    "VIT_Early_Conv": (
        ViTEarlyConvTransformer(img_size=32, patch_size=4, drop_path_rate=0.1).cuda()
    ),
}

all_histories = {}
torch.cuda.empty_cache()

for name, model in configs.items():
    print(f"\nUruchamianie treningu: {name}")
    model = model.cuda()
    loss = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

    if name == "CNN":
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, steps_per_epoch=len(train_dl), epochs=EPOCHS)
    else:
        optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
        scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, steps_per_epoch=len(train_dl), epochs=EPOCHS)

    history = fit(model, optimizer, scheduler, loss, train_dl, test_dl, epochs=EPOCHS)
    all_histories[name] = history

plot_metrixes(all_histories)


Uruchamianie treningu: CNN


Epoch 1/50: 100%|██████████| 782/782 [00:52<00:00, 14.84it/s]


Val Loss: 1.9114 | Val Acc: 0.3686


Epoch 2/50: 100%|██████████| 782/782 [00:54<00:00, 14.24it/s]


Val Loss: 1.7961 | Val Acc: 0.4229


...

Epoch 15/50: 100%|██████████| 782/782 [00:55<00:00, 14.09it/s]


Val Loss: 1.2875 | Val Acc: 0.6732


Epoch 16/50: 100%|██████████| 782/782 [00:51<00:00, 15.06it/s]


Val Loss: 1.2910 | Val Acc: 0.6606


Epoch 17/50: 100%|██████████| 782/782 [00:50<00:00, 15.34it/s]
